In [1]:
!pip install groq python-dotenv numpy tqdm datasets math-verify

In [2]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset, concatenate_datasets

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any, Optional

load_dotenv()
random.seed(0)

client = Groq()

MODEL = "llama-3.1-8b-instant"

/home/fling/ybigta-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### MATH 데이터셋 불러오기

- 평가: `HuggingFaceH4/MATH-500`
- few-shot 예시: `HuggingFaceH4/MATH`의 과목별 train split


In [3]:
# MATH 데이터 난이도 및 과목 필터링
TARGET_LEVELS = [1, 2, 3]

TARGET_SUBJECTS = [
    "Algebra",
    "Intermediate Algebra",
    "Number Theory",
    "Counting & Probability",
]

# 평가용 MATH-500
math_dataset = load_dataset("HuggingFaceH4/MATH-500")
math_test_raw = math_dataset["test"]

# few-shot 예시용 MATH train
TRAIN_CONFIGS = [
    "algebra",
    "intermediate_algebra",
    "number_theory",
    "counting_and_probability",
]

train_parts = []

for config in TRAIN_CONFIGS:
    ds = load_dataset(
        "HuggingFaceH4/MATH",
        config,
        split="train"
    )
    train_parts.append(ds)

math_train_raw = concatenate_datasets(train_parts)

print(sorted(set(math_train_raw["type"])))
print("raw train size:", len(math_train_raw))


Generating test split: 100%|██████████| 474/474 [00:00<00:00, 28242.06 examples/s]

['Algebra', 'Counting & Probability', 'Intermediate Algebra', 'Number Theory']
raw train size: 4679


In [4]:
## 데이터셋 전처리
def extract_last_boxed(text: str) -> Optional[str]:
    """문자열에서 마지막 \\boxed{...}의 내용을 추출합니다."""
    if not text:
        return None

    starts = [m.start() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return None

    start = starts[-1]
    open_brace = text.find("{", start)
    depth = 0

    for idx in range(open_brace, len(text)):
        if text[idx] == "{":
            depth += 1
        elif text[idx] == "}":
            depth -= 1
            if depth == 0:
                return text[open_brace + 1:idx].strip()

    return None


def parse_level(level_value) -> Optional[int]:
    match = re.search(r"\d+", str(level_value))
    return int(match.group()) if match else None


def prepare_math_train_row(row):
    return {
        "question": row["problem"],
        "answer": extract_last_boxed(row["solution"]),
        "rationale": row["solution"],
        "subject": row["type"],
        "level_num": parse_level(row["level"]),
    }


math_train = math_train_raw.map(prepare_math_train_row)

math_train = math_train.filter(
    lambda row: (
        row["level_num"] in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
        and row["answer"] is not None
    )
)

math_test = math_test_raw.filter(
    lambda row: (
        parse_level(row["level"]) in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
    )
)

print("math_train size:", len(math_train))
print("math_test size:", len(math_test))
print("train levels:", sorted(set(math_train["level_num"])))
print("test levels:", sorted(set(parse_level(x) for x in math_test["level"])))


Filter: 100%|██████████| 500/500 [00:00<00:00, 34458.06 examples/s]


math_train size: 2132
math_test size: 146
train levels: [1, 2, 3]
test levels: [1, 2, 3]


In [31]:
# ===== API 예산 관리 =====
import json, hashlib, time
from pathlib import Path
from collections import deque

TPM_LIMIT = 6000
TPM_SAFETY = 0.80          # 추정 오차 대비 여유
MAX_OUTPUT_TOKENS = 900    # 출력 폭주 방지
CACHE_PATH = Path("llm_cache.jsonl")

_cache = {}
_token_log = deque()       # (timestamp, tokens) 슬라이딩 윈도우
_daily = {"tokens": 0, "calls": 0, "cache_hits": 0}


def _load_cache():
    _cache.clear()
    if CACHE_PATH.exists():
        with open(CACHE_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    _cache[rec["key"]] = rec["response"]
                except json.JSONDecodeError:
                    continue
    print(f"캐시 로드: {len(_cache)}건")


def _cache_key(prompt: str, model: str) -> str:
    return hashlib.sha256(f"{model}||{prompt}".encode("utf-8")).hexdigest()


def _cache_put(key: str, response: str) -> None:
    _cache[key] = response
    with open(CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "response": response},
                           ensure_ascii=False) + "\n")


def estimate_tokens(text: str) -> int:
    """LaTeX는 토큰 밀도가 높아서 보수적으로 3자/토큰."""
    return len(text) // 3 + 8


def _throttle(planned: int) -> None:
    """최근 60초 사용량이 한도를 넘지 않도록 대기."""
    while True:
        now = time.time()
        while _token_log and now - _token_log[0][0] > 60:
            _token_log.popleft()
        used = sum(t for _, t in _token_log)
        if used + planned <= TPM_LIMIT * TPM_SAFETY or not _token_log:
            return
        wait = 60 - (now - _token_log[0][0]) + 0.5
        time.sleep(max(wait, 1.0))


def budget_report() -> None:
    print(f"누적 토큰: {_daily['tokens']:,} / 500,000  "
          f"({_daily['tokens'] / 5000:.1f}%)")
    print(f"API 호출: {_daily['calls']}건 | 캐시 적중: {_daily['cache_hits']}건")


_load_cache()


캐시 로드: 315건


In [32]:
MAX_OUTPUT_TOKENS = 900
print("MAX_OUTPUT_TOKENS =", MAX_OUTPUT_TOKENS)

MAX_OUTPUT_TOKENS = 900


In [12]:
# CoT few-shot용: 풀이가 짧고 그림(asy) 없는 예시만
MAX_SOLUTION_CHARS = 700

math_train_short = math_train.filter(
    lambda row: (
        row["rationale"] is not None
        and len(row["rationale"]) <= MAX_SOLUTION_CHARS
        and "[asy]" not in row["question"]
        and "[asy]" not in row["rationale"]
    )
)

print("filtered train size:", len(math_train_short))

filtered train size: 1947


In [13]:
def generate_response_using_Llama(prompt: str, model: str = MODEL):
    key = _cache_key(prompt, model)
    if key in _cache:
        _daily["cache_hits"] += 1
        return _cache[key]

    planned = estimate_tokens(prompt) + MAX_OUTPUT_TOKENS

    for attempt in range(6):
        _throttle(planned)
        try:
            comp = client.chat.completions.create(
                messages=[
                    {"role": "system",
                     "content": "You are a helpful assistant that solves math problems."},
                    {"role": "user", "content": prompt},
                ],
                model=model,
                temperature=0.0,
                max_tokens=MAX_OUTPUT_TOKENS,
                stream=False,
            )
            content = comp.choices[0].message.content
            usage = getattr(comp, "usage", None)
            actual = getattr(usage, "total_tokens", None) or planned

            _token_log.append((time.time(), actual))
            _daily["tokens"] += actual
            _daily["calls"] += 1
            _cache_put(key, content)
            return content

        except Exception as e:
            msg = str(e)
            if "429" in msg or "rate_limit" in msg.lower():
                m = re.search(r"try again in ([\d.]+)s", msg)
                wait = float(m.group(1)) + 1 if m else min(60, 5 * 2 ** attempt)
                print(f"[rate limit] {wait:.1f}초 대기 (재시도 {attempt + 1}/6)")
                time.sleep(wait)
                continue
            print(f"API error: {msg[:200]}")
            time.sleep(3)

    print("6회 재시도 실패 — None 반환")
    return None

#### 응답 잘 나오는지 확인하기

In [14]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### MATH 데이터셋 확인하기

In [7]:
print("[Question]")
print(math_test[0]["problem"])
print("=" * 100)
print("[Answer]")
print(math_test[0]["answer"])
print("=" * 100)
print("[Solution]")
print(math_test[0]["solution"])


[Question]
If $f(x) = \frac{3x-2}{x-2}$, what is the value of $f(-2) +f(-1)+f(0)$? Express your answer as a common fraction.
[Answer]
\frac{14}{3}
[Solution]
$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3(-1)-2}{-1-2}+\frac{3(0)-2}{0-2}=\frac{-8}{-4}+\frac{-5}{-3}+\frac{-2}{-2}=2+\frac{5}{3}+1=\boxed{\frac{14}{3}}$


#### Utils 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [18]:
def extract_final_answer(response: str):
    """응답에서 마지막 \\boxed{...} 또는 Answer: 뒤의 답을 추출합니다."""
    if response is None:
        return None

    boxed_answer = extract_last_boxed(response)
    if boxed_answer is not None:
        return boxed_answer

    matches = re.findall(
        r"(?:Final Answer|Answer)\s*:\s*(.+)",
        response,
        re.IGNORECASE
    )
    if matches:
        return matches[-1].strip().strip("$")

    return None


def normalize_math_text(text: Any) -> str:
    text = str(text).strip().strip("$")
    text = text.replace(r"\displaystyle", "")
    text = text.replace(r"\dfrac", r"\frac")
    text = text.replace(r"\tfrac", r"\frac")
    text = text.replace(r"\,", "")
    text = text.replace(" ", "")
    return text.rstrip(".")


try:
    from math_verify import parse, verify
    MATH_VERIFY_AVAILABLE = True
except Exception:
    MATH_VERIFY_AVAILABLE = False


def answers_equivalent(
    correct_answer: str,
    predicted_answer: Optional[str]
) -> bool:
    if predicted_answer is None:
        return False

    if MATH_VERIFY_AVAILABLE:
        try:
            correct_parsed = parse(f"${correct_answer}$")
            predicted_parsed = parse(f"${predicted_answer}$")

            if verify(correct_parsed, predicted_parsed):
                return True
        except Exception:
            pass

    return (
        normalize_math_text(correct_answer)
        == normalize_math_text(predicted_answer)
    )


print("math-verify available:", MATH_VERIFY_AVAILABLE)


math-verify available: True


In [19]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = MODEL,
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["problem"]
        correct_answer = str(dataset[i]["answer"]).strip()

        final_prompt = prompt.replace("{question}", question)

        response = generate_response_using_Llama(
            prompt=final_prompt,
            model=model
        )

        predicted_answer = (
            extract_final_answer(response)
            if response else None
        )
        is_correct = answers_equivalent(
            correct_answer,
            predicted_answer
        )

        if VERBOSE:
            print("=" * 50)
            print(response)
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {predicted_answer}")
            print(f"Correct: {is_correct}")
            print("=" * 50)

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(dataset[i]["level"]),
            "response": response,
        })

        if total % 5 == 0:
            current_accuracy = correct / total
            print(f"Progress: [{total}/{min(num_samples, len(dataset))}]")
            print(f"Current Acc.: [{current_accuracy:.2%}]")

    accuracy = correct / total if total > 0 else 0.0
    return results, accuracy


In [20]:
def save_final_result(
    results: List[Dict[str, Any]],
    accuracy: float,
    filename: str
) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += "[Details]\n"

    for idx, result in enumerate(results):
        result_str += f"Question {idx + 1}: {result['question']}\n"
        result_str += f"Subject: {result['subject']}\n"
        result_str += f"Level: {result['level']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)


#### 1. Direct Prompting with few-shot examples

In [21]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train

    rng = random.Random(1000 + num_examples)
    sampled_indices = rng.sample(range(len(train_dataset)), num_examples)
    

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question and generate ONLY the final answer "
        "after the tag 'Answer:' without any rationale. "
        "Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: {cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [22]:
### 어떤 방식으로 저장되는지 확인해보세요!

PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=math_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")
print(f"Direct 3-shot demo accuracy: {accuracy:.2%}")


 50%|█████     | 5/10 [00:03<00:03,  1.43it/s]

Progress: [5/10]
Current Acc.: [40.00%]


100%|██████████| 10/10 [00:24<00:00,  2.42s/it]

Progress: [10/10]
Current Acc.: [40.00%]
Direct 3-shot demo accuracy: 40.00%


In [23]:
import os

def run_and_save(prompt: str, filename: str, num_samples: int = 50):
    if os.path.exists(filename):
        print(f"[skip] {filename} 이미 있음")
        return
    print(f"\n===== {filename} (추정 입력 {estimate_tokens(prompt)} tok) =====")
    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=prompt,
        num_samples=num_samples,
        VERBOSE=False,
    )
    save_final_result(results, accuracy, filename)
    print(f"→ {filename}: {accuracy:.2%}")
    budget_report()

In [24]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!

for shot in [0, 3, 5]:
    run_and_save(construct_direct_prompt(shot), f"direct_prompting_{shot}.txt")


===== direct_prompting_0.txt (추정 입력 74 tok) =====


 10%|█         | 5/50 [00:02<00:18,  2.48it/s]

Progress: [5/50]
Current Acc.: [20.00%]


 20%|██        | 10/50 [00:04<00:15,  2.61it/s]

Progress: [10/50]
Current Acc.: [20.00%]


...

Progress: [40/50]
Current Acc.: [57.50%]


 90%|█████████ | 45/50 [07:22<00:36,  7.32s/it]

Progress: [45/50]
Current Acc.: [55.56%]


100%|██████████| 50/50 [08:13<00:00,  9.88s/it]

Progress: [50/50]
Current Acc.: [54.00%]
→ direct_prompting_5.txt: 54.00%
누적 토큰: 64,971 / 500,000  (13.0%)
API 호출: 151건 | 캐시 적중: 10건


#### 2. Chain-of-Thought Prompting with few-shot examples

```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 됩니다.

In [25]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train_short

    rng = random.Random(2000 + num_examples)
    sampled_indices = rng.sample(range(len(train_dataset)), num_examples)

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question. "
        "Reason step by step, keeping each step short and concrete. "
        "Then write the final answer inside \\boxed{} on the last line. "
        "Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: Let's think step by step.\n{cur_rationale}\n"
        prompt += f"The final answer is \\boxed{{{cur_answer}}}.\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [27]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!

for shot in [0, 3, 5]:
    run_and_save(construct_CoT_prompt(shot), f"CoT_prompting_{shot}.txt")


===== CoT_prompting_0.txt (추정 입력 87 tok) =====


 10%|█         | 5/50 [00:03<00:29,  1.51it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:07<00:28,  1.38it/s]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [70.00%]


 90%|█████████ | 45/50 [18:23<02:17, 27.47s/it]

Progress: [45/50]
Current Acc.: [71.11%]


100%|██████████| 50/50 [20:26<00:00, 24.53s/it]

Progress: [50/50]
Current Acc.: [70.00%]
→ CoT_prompting_5.txt: 70.00%
누적 토큰: 223,718 / 500,000  (44.7%)
API 호출: 301건 | 캐시 적중: 10건


#### 3. Construct your prompt + few shot examples
목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올리기!
- 세션때 배운 내용을 활용하거나 본인만의 풀이 과정을 만드는 등 자유롭게 진행해주시면 됩니다.
- 정답률은 Direct Prompting, CoT Prompting을 한 결과보다 높으면 됩니다. (0-shot, 3-shot, 5-shot 각각에서 모두 Direct Prompting과 CoT Prompting보다 높은 정답률을 달성하지 않더라도 감안하여 채점하겠습니다. 종합적으로 비교했을 때 본인이 설계한 프롬프트가 전반적으로 더 높은 성능을 보이는지를 기준으로 보겠습니다.)

In [43]:
import re, glob

def diagnose(path):
    txt = open(path, encoding="utf-8").read()
    blocks = txt.split("Question ")[1:]
    none_cnt = 0
    wrong_cnt = 0
    wrong_samples = []
    for b in blocks:
        pred = re.search(r"Predicted Answer: (.*)", b)
        corr = re.search(r"Correct: (\w+)", b)
        gold = re.search(r"Correct Answer: (.*)", b)
        if not (pred and corr):
            continue
        if corr.group(1) == "False":
            wrong_cnt += 1
            if pred.group(1).strip() == "None":
                none_cnt += 1
            elif len(wrong_samples) < 3:
                wrong_samples.append((gold.group(1)[:40], pred.group(1)[:40]))
    acc = re.search(r"ACCURACY: ([\d.]+)", txt).group(1)
    print(f"{path:28s} acc={float(acc):.0%} | 오답 {wrong_cnt} (파싱실패 {none_cnt}, 계산오류 {wrong_cnt - none_cnt})")
    for g, p in wrong_samples:
        print(f"     정답={g!r:42s} 예측={p!r}")


In [52]:
def construct_my_prompt(example_list = None, num_examples: int = 3):
    train_dataset = math_train_short

    rng = random.Random(3000 + num_examples)
    sampled_indices = rng.sample(range(len(train_dataset)), num_examples)

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question. Work step by step, taking as many "
        "steps as the problem needs.\n\n"
        "How to reason:\n"
        "- Every step must make progress. Never rewrite a line you have already "
        "written, and never re-simplify an expression that is already simplified.\n"
        "- Evaluate compactly: write 2^10 = 1024 rather than listing factors.\n"
        "- Before boxing, check the sign, the number of digits, and that no term is missing.\n\n"
        "Output rules (strict):\n"
        "- The last line must be exactly: \\boxed{ANSWER}\n"
        "- Write nothing after the \\boxed{} line.\n"
        "- Always output \\boxed{}, even if unsure or incomplete. Never end without it.\n"
        "- Simplify first: evaluate powers and roots, reduce fractions, rationalize "
        "denominators. Write \\frac{3}{2}, not 4^(3/2) or 1.5.\n"
        "- Match the requested form: common fraction, simplest radical form, "
        "base subscript, or text.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: Let's think step by step.\n{cur_rationale}\n"
        prompt += f"\\boxed{{{cur_answer}}}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [53]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
import os
if os.path.exists("My_prompting_0.txt"):
    os.remove("My_prompting_0.txt")
for shot in [0, 3, 5]:
    run_and_save(construct_my_prompt(num_examples=shot), f"My_prompting_{shot}.txt")


===== My_prompting_0.txt (추정 입력 298 tok) =====


 10%|█         | 5/50 [00:15<03:29,  4.66s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [01:16<06:34,  9.86s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [70.00%]


 90%|█████████ | 45/50 [16:19<02:16, 27.36s/it]

Progress: [45/50]
Current Acc.: [66.67%]


100%|██████████| 50/50 [17:25<00:00, 20.92s/it]

Progress: [50/50]
Current Acc.: [66.00%]
→ My_prompting_5.txt: 66.00%
누적 토큰: 272,742 / 500,000  (54.5%)
API 호출: 336건 | 캐시 적중: 71건


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot 정답률을 표로 보여주세요.
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요.
3. 본인이 작성한 프롬프트 기법에 대해서 설명하고 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요.
4. 위 내용들을 `PROMPTING.md`에 보고서로 작성해주세요.
